# 08 · Common Crawl on the Find a Tender sample (coverage)

Research log: [Step 9](../docs/research_log.md#step-9--common-crawl-pe-eșantionul-find-a-tender-măsurarea-acoperirii) (in Romanian). Data and checks as of October 2026.

**Question.** On a sample of active companies with a verified website, chosen without any link to a VAT source, how often does Common Crawl find the company's VAT number?

**Method.** Step-1 method (v3) on the 30-company Find a Tender sample: `CC-MAIN-2026-39`, max. 30 pages/domain, HTML + PDF; every valid number collected, then checked at HMRC; attribution via the HMRC name.

**Result (from the research log).**
- 25 companies with a result; domains in the crawl 19/25 (76%).
- VAT attributed to the exact company 3/25 (12%), 4/25 (16%) including a VAT-group case.
- 5/5 extracted numbers valid at HMRC, but 2 of 5 belonged to another entity (TLT (NI) LLP; Carter Jonas Service Company).

**Notes for re-running.**
- Calls the Common Crawl index (504s); ~46 s per company.
- The last cell (Carter Jonas lookup) needs the DuckDB connection defined in the setup cell.

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [ ]:
# Setup added for this notebook: copied verbatim from original cell(s) 66
import duckdb
CH_CSV = "../data/BasicCompanyDataAsOneFile-2026-09-01.csv"
con = duckdb.connect()


In [12]:
import io, re, json, time, html, csv
import requests
import pandas as pd
from warcio.archiveiterator import ArchiveIterator
from pypdf import PdfReader

CRAWL = "CC-MAIN-2026-39"          # același crawl ca în Pasul 1 și 2
MAX_PAGINI = 30
INCLUDE = {"da", "neverificat"}    # statusurile din domeniu_verificat_manual care intră în test
HEADERS = {"User-Agent": "veridion-vat-poc (student research)"}

es = pd.read_csv("../data/samples/fat_esantion_30_v3.csv", dtype=str, keep_default_na=False)
es = es[es.domeniu_verificat_manual.isin(INCLUDE)].reset_index(drop=True)
print("Firme în test:", len(es))

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

RE_VAT = re.compile(r"\bVAT\b[^0-9]{0,40}?(?:GB)?[\s\-]?((?:\d[\s\-]?){8}\d)(?![\s\-]?\d)", re.I)

def scor_prioritate(url):
    u = url.lower()
    if re.search(r"terms|condition|contact|legal|imprint|about|company-info", u): return 0
    if re.search(r"privacy|cookie|disclaimer", u): return 1
    if "polic" in u: return 2
    return 3

def get(url, **kw):
    """3 încercări. Întoarce răspunsul (200/206/404) sau None dacă serverul a dat eroare de fiecare dată."""
    for i in range(3):
        try:
            r = requests.get(url, headers={**HEADERS, **kw.pop("headers", {})}, timeout=120, **kw)
            if r.status_code in (200, 206, 404):
                return r
            print(f"    status {r.status_code}, reîncerc")
        except requests.RequestException as e:
            print(f"    eroare {type(e).__name__}, reîncerc")
        time.sleep(5 * (i + 1))
    return None

def pagini_din_index(domeniu):
    r = get(f"https://index.commoncrawl.org/{CRAWL}-index",
            params={"url": domeniu, "matchType": "domain", "output": "json"})
    if r is None:
        return None                      # eroare de server ≠ domeniu absent
    if r.status_code == 404:
        return []                        # domeniul nu e în crawl
    pagini, vazute = [], set()
    for linie in r.text.splitlines():
        p = json.loads(linie)
        mime = p.get("mime-detected") or p.get("mime", "")
        if p.get("status") == "200" and ("html" in mime or "pdf" in mime) and p["url"] not in vazute:
            vazute.add(p["url"])
            p["_pdf"] = "pdf" in mime
            pagini.append(p)
    pagini.sort(key=lambda p: (scor_prioritate(p["url"]), not p["_pdf"], len(p["url"])))
    return pagini

def text_pagina(p):
    start = int(p["offset"]); end = start + int(p["length"]) - 1
    r = get(f"https://data.commoncrawl.org/{p['filename']}", headers={"Range": f"bytes={start}-{end}"})
    if r is None:
        return None
    for rec in ArchiveIterator(io.BytesIO(r.content)):
        if rec.rec_type != "response":
            continue
        payload = rec.content_stream().read()
        if p["_pdf"]:
            try:
                reader = PdfReader(io.BytesIO(payload))
                return " ".join((pg.extract_text() or "") for pg in reader.pages[:20])
            except Exception:
                return "PDF_ILIZIBIL"
        t = payload.decode("utf-8", errors="ignore")
        t = re.sub(r"(?is)<(script|style).*?</\1>", " ", t)
        t = re.sub(r"<[^>]+>", " ", t)
        return re.sub(r"\s+", " ", html.unescape(t))
    return ""

def ch_in_context(ctx, ch):
    variante = {ch, ch.lstrip("0")} if ch.isdigit() else {ch}
    return any(re.search(rf"(?<![0-9A-Z]){re.escape(v)}(?![0-9])", ctx) for v in variante if v)

rez_firme, candidati = [], []
for _, f in es.iterrows():
    t0 = time.time()
    pagini = pagini_din_index(f.domeniu)
    if pagini is None:
        stare, citite, gasite = "EROARE_INDEX", 0, []
    elif not pagini:
        stare, citite, gasite = "ABSENT_DIN_CRAWL", 0, []
    else:
        citite, gasite = 0, []
        for p in pagini[:MAX_PAGINI]:
            text = text_pagina(p); citite += 1
            if not text:
                continue
            for m in RE_VAT.finditer(text):
                nr = re.sub(r"\D", "", m.group(1))
                if checksum_ok(nr):
                    ctx = text[max(0, m.start() - 200): m.end() + 200]
                    gasite.append(nr)
                    candidati.append({"company_number": f.company_number, "nume": f.nume,
                                      "domeniu": f.domeniu, "vat": nr, "url": p["url"],
                                      "pdf": p["_pdf"], "pozitie_in_lista": citite - 1,
                                      "ch_in_context": ch_in_context(ctx, f.company_number),
                                      "context": ctx})
            if gasite:
                break                    # ne oprim la prima pagină cu un TVA valid
            time.sleep(1)
        stare = "GASIT" if gasite else ("FARA_TVA_IN_PAGINI_CITITE")
    sec = round(time.time() - t0)
    rez_firme.append({"company_number": f.company_number, "nume": f.nume, "domeniu": f.domeniu,
                      "status_domeniu": f.domeniu_verificat_manual,
                      "pagini_in_index": None if pagini is None else len(pagini),
                      "pagini_citite": citite, "stare": stare,
                      "tva_gasite": " ".join(sorted(set(gasite))), "secunde": sec})
    print(f"{f.nume[:32]:32} {f.domeniu[:28]:28} index={'ERR' if pagini is None else len(pagini):>4} "
          f"citite={citite:>2} {stare:26} {' '.join(sorted(set(gasite)))}  {sec}s")
    time.sleep(2)

df_f = pd.DataFrame(rez_firme)
df_c = pd.DataFrame(candidati)
df_f.to_csv("../data/results/cc_fat_rezultat_firme.csv", index=False)
df_c.to_csv("../data/results/cc_fat_candidati.csv", index=False)
print("\n", df_f.stare.value_counts().to_string())
print("Timp total:", df_f.secunde.sum() // 60, "min")

Firme în test: 26
    status 504, reîncerc
Inspectas Compliance             inspectas.co.uk              index=   0 citite= 0 ABSENT_DIN_CRAWL             19s
Vilolet-Shift UK Ltd             violetshift.co.uk            index=   0 citite= 0 ABSENT_DIN_CRAWL             5s
AIION Ltd                        aiion.co.uk                  index=   9 citite= 9 FARA_TVA_IN_PAGINI_CITITE    25s
ICONI Software Ltd               iconi.co.uk                  index=   1 citite= 1 FARA_TVA_IN_PAGINI_CITITE    10s
ProofID                          proofid.com                  index=  41 citite=30 FARA_TVA_IN_PAGINI_CITITE    62s
    status 504, reîncerc
Ford Retail Limited              fordretail.com               index=  10 citite= 1 GASIT                      833053650  25s
THINKS INSIGHT AND STRATEGY (BRI thinksinsight.com            index=   0 citite= 0 ABSENT_DIN_CRAWL             10s
    status 504, reîncerc
    status 504, reîncerc
    status 504, reîncerc
TW ENGINEERING CO LIMITED        twen

EOF marker not found


Bytes Software Services Limited  bytes.co.uk                  index= 103 citite=30 FARA_TVA_IN_PAGINI_CITITE    75s
    status 504, reîncerc
TLT LLP                          tlt.com                      index= 679 citite= 1 GASIT                      129500920 752995876  27s
    status 504, reîncerc
    status 504, reîncerc
Holt Medical Recruitment Ltd     holtdoctors.co.uk            index=1197 citite=30 FARA_TVA_IN_PAGINI_CITITE    100s
    status 504, reîncerc
    status 504, reîncerc
    status 504, reîncerc
MRI Software Limited             mrisoftware.com              index= ERR citite= 0 EROARE_INDEX                 62s
    status 504, reîncerc
    status 504, reîncerc
    status 504, reîncerc
Bond Bryan Architects Limited    bondbryan.co.uk              index= ERR citite= 0 EROARE_INDEX                 62s
Devon Tree Services Limited      devontreeservices.co.uk      index=  53 citite=30 FARA_TVA_IN_PAGINI_CITITE    66s
Places for People Leisure Limite placesforpeopleleisure.org

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SNDABN+Helvetica', '/FontDescriptor': IndirectObject(12, 0, 127659283183760), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 32, '/Widths': [278], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/MWUNIP+Symbol', '/FontDescriptor': IndirectObject(14, 0, 127659283183760), '/Type': '/Font', '/FirstChar': 183, '/LastChar': 183, '/Widths': [460], '/Encoding': IndirectObject(58, 0, 127659283183760), '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SNDABN+Helvetica', '/FontDescriptor': IndirectObject(12, 0, 127659283183760), '/

NEC Software Solutions UK Limite necsws.com                   index= 149 citite=30 FARA_TVA_IN_PAGINI_CITITE    71s
Academia Limited                 academia.co.uk               index= 113 citite=30 FARA_TVA_IN_PAGINI_CITITE    71s

 stare
FARA_TVA_IN_PAGINI_CITITE    14
ABSENT_DIN_CRAWL              5
GASIT                         4
EROARE_INDEX                  3
Timp total: 20 min


In [13]:
# 1. Reluare pentru erorile de index
erori = df_f[df_f.stare == "EROARE_INDEX"].company_number.tolist()
es_retry = es[es.company_number.isin(erori)]
for _, f in es_retry.iterrows():
    time.sleep(10)
    pagini = pagini_din_index(f.domeniu)
    print(f.nume, "→", "ERR" if pagini is None else len(pagini), "pagini")
    if not pagini:
        df_f.loc[df_f.company_number == f.company_number, "stare"] = "EROARE_INDEX" if pagini is None else "ABSENT_DIN_CRAWL"
        continue
    gasite, citite = [], 0
    for p in pagini[:MAX_PAGINI]:
        text = text_pagina(p); citite += 1
        if not text:
            continue
        for m in RE_VAT.finditer(text):
            nr = re.sub(r"\D", "", m.group(1))
            if checksum_ok(nr):
                ctx = text[max(0, m.start() - 200): m.end() + 200]
                gasite.append(nr)
                candidati.append({"company_number": f.company_number, "nume": f.nume, "domeniu": f.domeniu,
                                  "vat": nr, "url": p["url"], "pdf": p["_pdf"], "pozitie_in_lista": citite - 1,
                                  "ch_in_context": ch_in_context(ctx, f.company_number), "context": ctx})
        if gasite:
            break
        time.sleep(1)
    df_f.loc[df_f.company_number == f.company_number, ["pagini_in_index", "pagini_citite", "stare", "tva_gasite"]] = \
        [len(pagini), citite, "GASIT" if gasite else "FARA_TVA_IN_PAGINI_CITITE", " ".join(sorted(set(gasite)))]
    print("   ", "GASIT " + " ".join(set(gasite)) if gasite else "fără TVA", f"({citite} pagini citite)")

df_c = pd.DataFrame(candidati)
df_f.to_csv("../data/results/cc_fat_rezultat_firme.csv", index=False)
df_c.to_csv("../data/results/cc_fat_candidati.csv", index=False)
print("\n", df_f.stare.value_counts().to_string())

# 2. Contextul pentru fiecare număr găsit
pd.set_option("display.max_colwidth", 400)
for _, c in df_c.drop_duplicates(["company_number", "vat"]).iterrows():
    print(f"\n{c.nume} | {c.vat} | CH în context: {c.ch_in_context} | {c.url}")
    print("   ", c.context[:400])

TW ENGINEERING CO LIMITED → 0 pagini
    status 504, reîncerc
    status 504, reîncerc
    status 504, reîncerc
MRI Software Limited → ERR pagini
Bond Bryan Architects Limited → 129 pagini
    fără TVA (30 pagini citite)

 stare
FARA_TVA_IN_PAGINI_CITITE    15
ABSENT_DIN_CRAWL              6
GASIT                         4
EROARE_INDEX                  1

Ford Retail Limited | 833053650 | CH în context: True | https://www.fordretail.com/terms-and-conditions/suppliers/
     designed by Bluesky Interactive. Registered in England. Registered Company Address: Ford Retail Ltd - 2 Charter Court, Newcomen Way, Colchester, Essex, CO4 9YA Registered In: United Kingdom - 191596 VAT Reg. No. 833053650. Ford Retail Group Ltd - 2 Charter Court, Newcomen Way, Colchester, Essex, CO4 9YA. Registered In: United Kingdom - 950395 Terms & Conditions Privacy Policy Site Map Modern Slav

TLT LLP | 752995876 | CH în context: False | https://www.tlt.com/legal-notices
     are not admitted as solicitors in Eng

In [14]:
con.execute(f"""
    SELECT companynumber, companyname, regaddresspostcode, companystatus, accountsaccountcategory
    FROM read_csv('{CH_CSV}', normalize_names=true, ignore_errors=true, all_varchar=true)
    WHERE companyname ILIKE '%CARTER JONAS%'
""").df()

,companynumber,companyname,regaddresspostcode,companystatus,accountsaccountcategory
0,08752910,CARTER JONAS ACQUISITIONS LTD,W1G 0BG,Active,FULL
1,09342548,CARTER JONAS ENTERPRISES LIMITED,W1G 0BG,Active,AUDIT EXEMPTION SUBSIDIARY
2,06287947,CARTER JONAS LIMITED,W1G 0BG,Active,DORMANT
3,OC304417,CARTER JONAS LLP,W1G 0BG,Active,GROUP
4,00786457,CARTER JONAS SERVICE COMPANY,W1G 0BG,Active,FULL
5,03207537,THE CARTER JONAS GROUP LIMITED,W1G 0BG,Active,DORMANT
